## Welcome to Lab 3 for Week 1 Day 4

Today we're going to build something with immediate value!

In the folder `me` I've put a single file `linkedin.pdf` - it's a PDF download of my LinkedIn profile.

Please replace it with yours!

I've also made a file called `summary.txt`

We're not going to use Tools just yet - we're going to add the tool tomorrow.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Looking up packages</h2>
            <span style="color:#00bfff;">In this lab, we're going to use the wonderful Gradio package for building quick UIs, 
            and we're also going to use the popular PyPDF PDF reader. You can get guides to these packages by asking 
            ChatGPT or Claude, and you find all open-source packages on the repository <a href="https://pypi.org">https://pypi.org</a>.
            </span>
        </td>
    </tr>
</table>

In [1]:
# If you don't know what any of these packages do - you can always ask ChatGPT for a guide!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader # For reading PDF files
import gradio as gr # For creating the web interface

In [2]:
load_dotenv(override=True)
openai = OpenAI()

In [3]:
reader = PdfReader("me/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [4]:
print(linkedin)

   
Contact
03013716896 (Mobile)
hasnainasif52@gmail.com
www.linkedin.com/in/hasnain-asif
(LinkedIn)
Top Skills
Team Leadership
Business Ownership
Mentoring
Languages
Urdu (Native or Bilingual)
English (Limited Working)
Punjabi (Native or Bilingual)
Hasnain Asif
JavaScript | TypeScript | ReactJs | NodeJs
Sādiqābād, Punjab, Pakistan
Summary
Hey there!  I am a Data Science enthusiast and a Full Stack
Developer with 5+ years of experience. I excel in Javascript,
Typescript, React, Redux, Styled-Components, Tailwind, Bootstrap,
Node.js, Express.js, Socket.io, MongoDB, Postgresql, REST APIs,
and GraphQL. 
My results speak volumes—I've led successful projects, from NFT
marketplaces to decentralized finance platforms. Worldwide users
and significant revenue prove my clean code and seamless user
experiences.
I'm a quick learner and excellent communicator, so I understand
your needs precisely. I thrive in fast-paced environments, delivering
incremental changes that get fast feedback. Collaborat

In [5]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [6]:
name = "Hasnain Asif"

In [7]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
If you don't know the answer, say so."

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."


In [8]:
system_prompt

"You are acting as Hasnain Asif. You are answering questions on Hasnain Asif's website, particularly questions related to Hasnain Asif's career, background, skills and experience. Your responsibility is to represent Hasnain Asif for interactions on the website as faithfully as possible. You are given a summary of Hasnain Asif's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Hasnain Asif. I'm a software engineer, MERN Stack Developer and data science learner. I'm from SadiqAbad, Pakistan.\nI love all halal foods, particularly Biryani. I love to read books and listen to my mentors.\n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\n03013716896 (Mobile)\nhasnainasif52@gmail.com\nwww.linkedin.com/in/hasnain-asif\n(LinkedIn)\nTop Skills\nTeam Leadership\nBusiness Ownership\nMentoring\nLanguages\nU

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    print(history)
    print('-' * 100)
    print(messages)
    model_name = "gpt-4.1-nano" # gpt-4o-mini
    response = openai.chat.completions.create(model=model_name, messages=messages)
    return response.choices[0].message.content

## Special note for people not using OpenAI

Some providers, like Groq, might give an error when you send your second message in the chat.

This is because Gradio shoves some extra fields into the history object. OpenAI doesn't mind; but some other models complain.

If this happens, the solution is to add this first line to the chat() function above. It cleans up the history variable:

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

You may need to add this in other chat() callback functions in the future, too.

In [ ]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


[]
----------------------------------------------------------------------------------------------------
[{'role': 'system', 'content': "You are acting as Hasnain Asif. You are answering questions on Hasnain Asif's website, particularly questions related to Hasnain Asif's career, background, skills and experience. Your responsibility is to represent Hasnain Asif for interactions on the website as faithfully as possible. You are given a summary of Hasnain Asif's background and LinkedIn profile which you can use to answer questions. Be professional and engaging, as if talking to a potential client or future employer who came across the website. If you don't know the answer, say so.\n\n## Summary:\nMy name is Hasnain Asif. I'm a software engineer, MERN Stack Developer and data science learner. I'm from SadiqAbad, Pakistan.\nI love all halal foods, particularly Biryani. I love to read books and listen to my mentors.\n\n## LinkedIn Profile:\n\xa0 \xa0\nContact\n03013716896 (Mobile)\nhasnaina

## A lot is about to happen...

1. Be able to ask an LLM to evaluate an answer
2. Be able to rerun if the answer fails evaluation
3. Put this together into 1 workflow

All without any Agentic framework!

`Workflow design pattern used here: Evaluator-Optimizer`

In [35]:
# Create a Pydantic model for the Evaluation

# --> Pydantic Model is: Using class to describe particular data structure of information - GPT response will be returned in this format

from pydantic import BaseModel

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str


In [36]:
evaluator_system_prompt = f"You are an evaluator that decides whether a response to a question is acceptable. \
You are provided with a conversation between a User and an Agent. Your task is to decide whether the Agent's latest response is acceptable quality. \
The Agent is playing the role of {name} and is representing {name} on their website. \
The Agent has been instructed to be professional and engaging, as if talking to a potential client or future employer who came across the website. \
The Agent has been provided with context on {name} in the form of their summary and LinkedIn details. Here's the information:"

evaluator_system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
evaluator_system_prompt += f"With this context, please evaluate the latest response, replying with whether the response is acceptable and your feedback."

In [37]:
def evaluator_user_prompt(reply, message, history):
    user_prompt = f"Here's the conversation between the User and the Agent: \n\n{history}\n\n"
    user_prompt += f"Here's the latest message from the User: \n\n{message}\n\n"
    user_prompt += f"Here's the latest response from the Agent: \n\n{reply}\n\n"
    user_prompt += "Please evaluate the response, replying with whether it is acceptable and your feedback."
    return user_prompt

In [ ]:
# ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama') # api_key could be anything for local models
# ollama_model_name = "llama3.2"

# import os
# deepseek_api_key = os.getenv('DEEPSEEK_API_KEY')
# deepseek = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com/v1")
# deepseek_model_name = "deepseek-chat"

# openai = OpenAI()
openai_model_name = "gpt-4o-mini"

In [45]:
def evaluate(reply, message, history) -> Evaluation:

    messages = [{"role": "system", "content": evaluator_system_prompt}] + [{"role": "user", "content": evaluator_user_prompt(reply, message, history)}]
    response = openai.beta.chat.completions.parse(model=openai_model_name, messages=messages, response_format=Evaluation) # Way to call api, to get structured outputs
    return response.choices[0].message.parsed

In [ ]:
# Ask question from openai, by providing initial System prompt and then a user question
messages = [{"role": "system", "content": system_prompt}] + [{"role": "user", "content": "do you hold a patent?"}]
response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
reply = response.choices[0].message.content

In [46]:
reply

"No, I do not currently hold a patent. My focus has been primarily on software development and contributing to impactful projects in the tech industry. If you have any questions regarding my skills or projects I've worked on, feel free to ask!"

In [47]:
evaluate(reply, "do you hold a patent?", messages[:1])

Evaluation(is_acceptable=True, feedback="The response is acceptable as it is direct, concise, and professionally addresses the user's question about patent ownership. The Agent also encourages further engagement by inviting additional questions related to their skills or projects, maintaining an open and engaging tone. This aligns well with the instructions to be professional and engaging while representing Hasnain Asif.")

In [48]:
def rerun(reply, message, history, feedback):
    updated_system_prompt = system_prompt + "\n\n## Previous answer rejected\nYou just tried to reply, but the quality control rejected your reply\n"
    updated_system_prompt += f"## Your attempted answer:\n{reply}\n\n"
    updated_system_prompt += f"## Reason for rejection:\n{feedback}\n\n"
    messages = [{"role": "system", "content": updated_system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
    return response.choices[0].message.content

In [ ]:
def chat(message, history):
    if "patent" in message:
        # if the user is asking about patents, respond in pig latin. - Its just a way to get is_acceptable as false
        system = system_prompt + "\n\nEverything in your reply needs to be in pig latin - \
              it is mandatory that you respond only and entirely in pig latin"
    else:
        system = system_prompt
    messages = [{"role": "system", "content": system}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-4.1-nano", messages=messages)
    reply =response.choices[0].message.content

    evaluation = evaluate(reply, message, history)
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
    else:
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        reply = rerun(reply, message, history, evaluation.feedback)       
    return reply

Passed evaluation - returning reply


In [56]:
gr.ChatInterface(chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


aaaaaaaaaaaaaaaaaa:  True
Passed evaluation - returning reply
